<a href="https://colab.research.google.com/github/quraishidev51/ECG-analyzer-report-generator/blob/main/notebooks/inferencev1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
#mounting drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
#finding path
!ls /content/drive/MyDrive/ECG_Project/

best_model.keras


In [30]:
#loading the model
from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/ECG_Project/best_model.keras')

In [31]:
#loading the dataset
import numpy as np
X = np.load("/content/drive/MyDrive/X_raw.npy")

In [32]:
print(X.shape)
print(X.dtype)

(21837, 1000, 12)
float64


In [33]:
pred = model.predict(X[:1])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [34]:
print(pred)

[[2.0682748e-01 4.4572282e-01 5.9883666e-01 1.5909778e-04 8.9677817e-01]]


In [35]:
CLASS_NAMES = ["Conduction Disturbance","Hypertrophy","Myocardial Infarction","Normal","ST/T Change"]
def predict_ecg(ecg_signal, threshold = 0.5):
      # Add batch dimension
    ecg_signal = np.expand_dims(ecg_signal, axis=0)

    probs = model.predict(ecg_signal, verbose=0)[0]

    probabilities = {
        cls: float(prob)
        for cls, prob in zip(CLASS_NAMES, probs)
    }

    predictions = [
        {
            "label": cls,
            "confidence" : float(prob)
        }
        for cls, prob in probabilities.items()
        if prob >= threshold
    ]

    return {
    "probabilities": probabilities,
    "predictions": predictions
}

In [36]:
predict_ecg(X[0])

{'probabilities': {'Conduction Disturbance': 0.20682747662067413,
  'Hypertrophy': 0.4457228183746338,
  'Myocardial Infarction': 0.5988366603851318,
  'Normal': 0.00015909777721390128,
  'ST/T Change': 0.8967781662940979},
 'predictions': [{'label': 'Myocardial Infarction',
   'confidence': 0.5988366603851318},
  {'label': 'ST/T Change', 'confidence': 0.8967781662940979}]}

In [53]:
def create_report(result):
  report = []#appending lines
  report.append("ECG Analysis Report")
  report.append("-"*30)
  report.append("")
  report.append("Detected Conditions:")

  for prediction in sorted(
      result["predictions"],
      key=lambda x: x["confidence"],
      reverse=True
  ):
    label = prediction["label"]
    confidence = prediction["confidence"]
    report.append(
        f"✓ {label} ({confidence*100:.1f}%)"
    )

  predicted = {
    prediction["label"]
    for prediction in result["predictions"]
}

  report.append("")
  report.append("Other probabilities:")

  for label, prob in sorted(
    result["probabilities"].items(),
    key=lambda item: item[1],
    reverse=True
):
    if label not in predicted:
        report.append(
            f"{label: <28}: {prob*100:.1f}%"
        )
  return "\n".join(report)

In [54]:
result = predict_ecg(X[0])
report = create_report(result)
print(report)

ECG Analysis Report
------------------------------

Detected Conditions:
✓ ST/T Change (89.7%)
✓ Myocardial Infarction (59.9%)

Other probabilities:
Hypertrophy                 : 44.6%
Conduction Disturbance      : 20.7%
Normal                      : 0.0%
